In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_prompt=None, stop_sequences=None):
    parameters = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system_prompt:
        parameters["system"] = system_prompt

    if stop_sequences:
        parameters["stop_sequences"] = stop_sequences

    message = client.messages.create(**parameters)
    return message.content[0].text if getattr(message.content[0], 'text', None) else message.content[1].text

In [3]:
import json
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
    "solution_criteria": "Description of what constitutes a correct solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(dataset)

[{'task': "Parse an AWS S3 bucket name and object key from an S3 URI in the format 's3://bucket-name/path/to/object'. Extract both the bucket name and the full object key.", 'format': 'regex', 'solution_criteria': 'The regex should correctly capture the bucket name and object key from valid S3 URIs. It should handle bucket names with hyphens and numbers, and object keys with multiple path segments.'}, {'task': "Write a Python function that takes an AWS CloudFormation template as a dictionary and returns a list of all logical resource IDs that have a resource type starting with 'AWS::Lambda'.", 'format': 'python', 'solution_criteria': 'The function should iterate through the Resources section of the template, identify all Lambda-related resources, and return their logical IDs as a list. It should handle cases where the Resources section is missing or empty.'}, {'task': "Create a JSON object representing an AWS IAM policy that grants read-only access to a specific S3 bucket named 'my-dat

In [5]:
with open('dataset-exercise.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
def grade_by_model(test_case, output):
    try:
        # Create evaluation prompt
        eval_prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        
        Task: {test_case.get('task', 'No task provided')}
        Solution: {output}
        Sulution criteria: {test_case.get('solution_criteria', 'No specific criteria provided')}
        
        Provide your evaluation as a structured JSON object with:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement  
        - "reasoning": A concise explanation of your assessment
        - "score": A number between 1-10
        """
        
        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        
        eval_text = chat(messages, stop_sequences=["```"])
        return json.loads(eval_text)
    except Exception as e:
        print(f"Error in grade_by_model: {e} - eval_text: {eval_text}")

In [7]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(output, test_case):
    format = test_case.get("format")
    if format == "json":
        return validate_json(output)
    elif format == "python":
        return validate_python(output)
    elif format == "regex":
        return validate_regex(output)
    else:
        return 0

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [9]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)
    reasoning = model_grade["reasoning"]
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output, 
        "test_case": test_case, 
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }

In [10]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [13]:
with open("dataset-exercise.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.333333333333333


In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\ndef parse_s3_uri(s3_uri):\n    \"\"\"\n    Parse an AWS S3 URI and extract bucket name and object key.\n    \n    Args:\n        s3_uri (str): S3 URI in format 's3://bucket-name/path/to/object'\n    \n    Returns:\n        tuple: (bucket_name, object_key) or (None, None) if invalid\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not isinstance(s3_uri, str):\n        raise ValueError(\"S3 URI must be a string\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Find the first '/' to separate bucket from key\n    first_slash_index = uri_without_prefix.find('/')\n    \n    if first_slash_index == -1:\n        # No object key provided, only bucket\n        bucket_name = uri_without_prefix\n        object_key = ''\n    else:\n        bucket_name = uri_without_prefix[:first_slash_index]\n